In [1]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.5 MB/s eta 0:00:00


In [2]:
import os
import numpy as np

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

DATA_PATH = "/content/drive/MyDrive/KnowledgeHub_RAG/data"

pdf_files = [
    os.path.join(DATA_PATH, file)
    for file in os.listdir(DATA_PATH)
    if file.lower().endswith(".pdf")
]

print(f"Found {len(pdf_files)} PDF(s):")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

Found 2 PDF(s):
- Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
- SurendaranathK_Presentation.pdf


In [5]:
def load_pdf(pdf_path):
    """
    Extract text from a PDF.
    """
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text


documents = []

for pdf in pdf_files:

    text = load_pdf(pdf)

    documents.append(
        {
            "filename": os.path.basename(pdf),
            "text": text
        }
    )

print(f"Loaded {len(documents)} document(s).")

Loaded 2 document(s).


In [6]:
def chunk_text(text, chunk_size=1000, overlap=200):

    paragraphs = text.split("\n\n")

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) <= chunk_size:
            current_chunk += paragraph + "\n\n"
        else:
            chunks.append(current_chunk.strip())

            current_chunk = current_chunk[-overlap:] + paragraph + "\n\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [7]:
all_chunks = []

for document in documents:

    chunks = chunk_text(document["text"])

    for chunk in chunks:

        all_chunks.append(
            {
                "document": document["filename"],
                "text": chunk
            }
        )

print(f"Created {len(all_chunks)} chunks.")

Created 5 chunks.


In [8]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)
print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(5, 384)


In [10]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

5


In [11]:
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    results = []

    for idx, score in zip(indices[0], distances[0]):

        results.append({
            "document": all_chunks[idx]["document"],
            "text": all_chunks[idx]["text"],
            "score": float(score)
        })

    return results

In [12]:
results = retrieve(
    "What algorithm was used for classification?",
    top_k=5
)

for r in results:

    print("="*80)
    print(r["score"])
    print(r["document"])
    print(r["text"][:500])

0.34583863615989685
SurendaranathK_Presentation.pdf
M.Sc - Artificial Intelligence and Machine Learning in Science
S P C 7 2 0 P  
R E S E A R C H  P R O J E C T  I N  D A T A  S C I E N C E  
Surendaranath Kanniyappan
 Supervisor : Dr. Michalis Agathos
MA C H I N E  L E A R N I N G  C L A S S I F I C A T I O N
O F  B I N A R Y  N E U T R O N  S T A R  R E MN A N T S
U S I N G  G R A V I T A T I O N A L  WA V E  D A T A
0.15589343011379242
SurendaranathK_Presentation.pdf
t curves, GRB energetics,
neutrinos).
Develop a low-latency pipeline for real-time GW alerts.
Extend framework to future GW detectors (Einstein Telescope, Cosmic Explorer).
FUTURE WORK
11
THANK YOU!
0.13237309455871582
Surendaranath_Kanniyappan_FinalProjectReport_SPC720P (1).pdf
Machine Learning Classification of Binary Neutron
Star Remnants Using Gravitational Wave Data
Surendaranath Kanniyappan
Dr. Michalis Agathos
Abstract
Binary neutron star (BNS) mergers are among the most energetic cosmic events,
producing gravit

In [13]:
faiss.write_index(
    index,
    "knowledgehub.index"
)

In [14]:
loaded_index = faiss.read_index(
    "knowledgehub.index"
)

print(loaded_index.ntotal)

5


In [15]:
import time

query = "What algorithm was used for classification?"

start = time.time()

retrieve(query)

end = time.time()

print(f"FAISS Retrieval Time: {end-start:.6f} seconds")

FAISS Retrieval Time: 0.048779 seconds
